# 05 · Labelled suspicious-activity analytics ("fraud analytics" phase)
**Scope note:** AMLSim has **no payment-fraud label**, no card/merchant/device/channel fields and **no loss amounts**.
Its only label marks accounts that participate in an injected AML typology. We therefore apply fraud-analytics
*methods* - group comparison, pre-specified hypothesis tests, unsupervised anomaly detection and value proxies -
to that label, and we never estimate losses.

In [1]:
import sys, json, sqlite3
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30); pd.set_option("display.precision", 4)
T = lambda name: pd.read_csv(config.TABLES_DIR / f"{name}.csv")
KM = json.loads((config.TABLES_DIR / "key_metrics.json").read_text())
con = sqlite3.connect(config.DB_PATH)
def show(fig_name, width=11):
    img = plt.imread(config.FIGURES_DIR / f"{fig_name}.png")
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h / w)); ax.imshow(img); ax.axis("off")
print("outputs loaded from", config.TABLES_DIR.relative_to(ROOT))

outputs loaded from outputs/tables


### Pre-specified hypotheses (all reported, Holm-corrected)
* H1 amounts differ · H2 higher velocity · H3a/b more distinct senders/receivers · H3c more cycle participation
* H4 integrated score detects labelled accounts the amount rule misses · H5 integrated ranking beats amount+velocity

Assumptions: Mann-Whitney U assumes independent observations (accounts are the unit, not transactions);
Fisher's exact test for 2x2 counts; McNemar for paired detection; H5 uses a paired account bootstrap.

In [2]:
h = T("hypothesis_tests")
h[["hypothesis", "description", "test", "n_group1", "n_group2", "median_group1", "median_group2", "rate_group1",
   "rate_group2", "effect_size", "effect_size_name", "ci_low", "ci_high", "p_holm"]]

,hypothesis,description,test,n_group1,n_group2,median_group1,median_group2,rate_group1,rate_group2,effect_size,effect_size_name,ci_low,ci_high,p_holm
0,H1,Labelled accounts' typical (median) transfer amount differs from unlabelled accounts',Mann-Whitney U (two-sided),1798,17969,222.73,254.8300,NaN,NaN,-0.1435,rank-biserial r,-39.3301,-23.8650,1.8702e-23
1,H2,Labelled accounts transact at higher velocity (transfers per active week),"Mann-Whitney U (one-sided, greater)",1798,17969,1.80,1.3333,NaN,NaN,0.4436,rank-biserial r,0.4167,0.5000,3.6081e-218
2,H3a,Labelled accounts have more distinct senders in their busiest 28-day window,"Mann-Whitney U (one-sided, greater)",1798,17969,3.00,2.0000,NaN,NaN,0.2954,rank-biserial r,1.0000,1.0000,4.4289e-98
3,H3b,Labelled accounts have more distinct receivers in their busiest 28-day window,"Mann-Whitney U (one-sided, greater)",1798,17969,3.00,2.0000,NaN,NaN,0.2746,rank-biserial r,1.0000,1.0000,1.1410e-86
4,H3c,Labelled accounts participate in time-respecting cycles more often,Fisher exact (two-sided),1804,18176,NaN,NaN,0.0377,0.0023,16.9123,odds ratio,11.4759,24.9242,9.4190e-43
5,H4,"Among labelled accounts, the integrated score detects cases the amount rule misses","Exact McNemar (paired, labelled held-out accounts)",120,46,NaN,NaN,NaN,NaN,3.6429,discordant ratio (A only / B only),NaN,NaN,4.4254e-11
6,H5,Integrated score ranks labelled accounts better than amount+velocity (PR-AUC),"Paired account bootstrap (1,000 resamples) of PR-AUC difference",9277,9277,NaN,NaN,NaN,NaN,0.0123,PR-AUC difference,0.0045,0.0198,NaN


In [3]:
# statistical vs practical significance
for k, r in h.set_index("hypothesis").iterrows():
    print(f"{k}: effect {r.effect_size:.3f} ({r.effect_size_name}); estimate {r.estimate:.4g} [{r.ci_low:.4g}, {r.ci_high:.4g}]")

H1: effect -0.144 (rank-biserial r); estimate -32.1 [-39.33, -23.87]
H2: effect 0.444 (rank-biserial r); estimate 0.4667 [0.4167, 0.5]
H3a: effect 0.295 (rank-biserial r); estimate 1 [1, 1]
H3b: effect 0.275 (rank-biserial r); estimate 1 [1, 1]
H3c: effect 16.912 (odds ratio); estimate 0.03538 [11.48, 24.92]
H4: effect 3.643 (discordant ratio (A only / B only)); estimate 74 [nan, nan]
H5: effect 0.012 (PR-AUC difference); estimate 0.01232 [0.004463, 0.01983]


### Unsupervised anomaly detection (Isolation Forest) vs the label

In [4]:
rk = T("ranking_metrics").set_index("approach")
rk.loc[["ML3 Isolation Forest (unsupervised)", "B2 Amount + velocity", "B5b Integrated risk score (top 1%)"],
       ["roc_auc", "pr_auc", "precision_top5pct", "recall_top5pct", "base_rate"]]

,roc_auc,pr_auc,precision_top5pct,recall_top5pct,base_rate
approach,,,,,
ML3 Isolation Forest (unsupervised),0.8086,0.3985,0.5431,0.2954,0.0919
B2 Amount + velocity,0.7834,0.4450,0.6875,0.3740,0.0919
B5b Integrated risk score (top 1%),0.7534,0.4573,0.6875,0.3740,0.0919


### Measurable value proxies (not losses)

In [5]:
T("value_proxies")

,measure,value,share_of_total
0,total transfer value,3.3288e+07,1.0000
1,value between two labelled accounts (typology proxy),1.2521e+06,0.0376
2,value touching >= 1 labelled account,9.2900e+06,0.2791
3,transfers between two labelled accounts,7.8850e+03,0.0654
4,"mean amount, typology-proxy transfers",1.5880e+02,NaN
5,"mean amount, other transfers",2.8433e+02,NaN
